In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model # pyright: ignore[reportMissingModuleSource]
import pickle
import pandas as pd
import numpy as np

In [2]:
#Loading the ANN trained model, Scaler pickle file, Geography and Gender pickle files
model = load_model('model.h5')

with open('label_encoder_gender.pkl','rb') as file:
    loaded_label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    loaded_onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [3]:
#Example input data
input_data = {
    'CreditScore': 500,
    'Geography': 'France',
    'Gender':'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [7]:
#Onehot encoding 'Geography'
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoded = loaded_onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = loaded_onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\KIIT\Downloads\PythonNew\venv_w2v\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [9]:
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,500,France,Male,40,3,60000,2,1,1,50000


In [10]:
input_data_df['Gender'] = loaded_label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,500,France,1,40,3,60000,2,1,1,50000


In [11]:
#concatenation with original data
input_data_df = pd.concat([input_data_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,500,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [16]:
#scaling the input data
input_data_df_scaled = scaler.transform(input_data_df)
input_data_df_scaled

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [19]:
#predict churn modelling
prediction = model.predict(input_data_df_scaled)
prediction_probability = prediction[0][0]
if prediction_probability > 0.5:
    print("Person likely to churn")
else:
    print("person unlikely to churn")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
person unlikely to churn
